# Laboration 3: Optimising a Catalytic Reaction

**Team name:**
**Members:**
**Date:**

Follow `instruction.md` for the full task description, timing, and the
Checkpoint questions. This notebook is your **working notebook** — light
comments are enough here; your answers to the Checkpoints go on a
**separate answer sheet**, not in this file.

Before you start: open `plant_client.py` and fill in `SERVER_URL`,
`TEAM`, and `TOKEN` (given to you by your instructor).

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import qmc, t as t_dist
from scipy import stats
from scipy.optimize import minimize
import itertools

from plant_client import run_experiment, leaderboard

RANDOM_STATE = 42

# Keep every experiment you run here -- you'll need the full history in
# Step 3, not just your latest batch.
history = pd.DataFrame(columns=["T", "c", "rpm", "rate"])

def log(result):
    """Append a run_experiment() result to the running history table."""
    global history
    history = pd.concat([history, result], ignore_index=True)
    return history

---
## Step 1 — Factorial Screening (0:00–0:45)

Three candidate factors this time: **c**, **T**, and **stirring rate
(rpm)** — nobody has confirmed rpm actually matters, that's what this
step is for.

- [ ] Design a 2³ factorial (8 corner points) around the OVAT point (c=2.0, T=25.0) and a sensible centre rpm, plus a few centre-point replicates
- [ ] Run it via `run_experiment`
- [ ] Code each factor to +/-1 (e.g. `c_coded = (c - 2.0) / 0.5`) before fitting -- see the TODO below for why this matters
- [ ] Fit `rate ~ c_coded * T_coded * rpm_coded` on the corner points — which effects are significant? (→ **Checkpoint 1a**)
- [ ] If rpm (and its interactions) aren't significant, drop it and fix it at a convenient value from here on
- [ ] Compare centre-point average to corner-point average — is there curvature? (→ **Checkpoint 1b**)

In [21]:
# TODO:
# 1. choose your factor ranges (e.g. c +/- 0.5 mM, T +/- 5 degC around
# the OVAT point, rpm +/- 200 around a centre value like 500)
# 2. build the 8 corner points (save in a df)
# 3. Build a few centre-point repeats (save in a separate df)
C_center = [2,2]
T_center = [25,25]
rpm_center = [500,500]

# HINT: itertools.product(levels_c, levels_T, levels_rpm)
# is the easiest way to build the 2^3 corners.

In [24]:
# TODO:
# 1. run the design with result = run_experiment(name="...", T=..., c=..., rpm=...).
# You will have to run the corner points and centre-points
# separately (it is needed for the next parts of the lab)
# 2. Use log(result) to add it to `history`.
result = run_experiment(T = T_center, c = C_center, rpm = rpm_center, name= 'center_run_1')
result
log(result)


# TIP: the run_experiment function is defined in plant_client.py.

ConnectionError: HTTPConnectionPool(host='%3cinstructor-ip-goes-here%3e', port=5000): Max retries exceeded with url: /experiment (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7ead9f775d10>: Failed to resolve '%3cinstructor-ip-goes-here%3e' ([Errno -2] Name or service not known)"))

In [ ]:
# TODO:
# 1. For the result, code each factor to +/-1 using the factorial's own half-ranges
# (e.g. c_coded = (c - 2.0) / 0.5) and add these as new columns.
# 2. THEN fit rate ~ c_coded * T_coded * rpm_coded (statsmodels.formula.api.ols)
# ONLY on the corner points. (Fitting on raw units (mM, degC, rpm) instead will give
# you a badly ill-conditioned model where every effect looks non-significant
# that's a red flag to code your variables, not a real finding.)

# TIP: .summary() shows you the coefficients, signs, and p-values for every
# main effect and interaction.

In [ ]:
# TODO: decide whether to drop rpm based on the model above. If you do,
# note it here (and on the answer sheet, Checkpoint 1a) and pick a fixed
# rpm value to use for the rest of the lab.

In [ ]:
# TODO: compare the centre-point average rate to the average of the 8
# corner points. A meaningfully different value suggests curvature.
# Reason about this on the answer sheet (Checkpoint 1b), not just here.

---
## Step 2 — Steepest Ascent (0:45–1:30)

- [ ] Use the significant main effects from Step 1 (c and T, assuming rpm was dropped) to compute an ascent direction
- [ ] Run a sequence of points stepping along that direction, until the rate stops improving. If you fixed the repm value, keep it fixed at your chosen value.
- [ ] Keep `log()`-ing every result so `history` stays complete

In [ ]:
# TODO:
# 1. Compute the steepest-ascent direction from your Step 1 model's c
# and T coefficients (the ratio of the fitted effects gives the relative
# step sizes)
# 2. Choose a step size and take your first step. If you fixed
# your rpm value, pass it on every call.

In [ ]:
# TODO:
# Perform a steepest ascent along the ascent direction, starting from the OVAT point.
# Run the experiments using result = run_experiment(name, T, c, rpm)
# and log each result using log(result).
# Print the best (T, c, rate) you've found so far after each step.
# Continue the steepest ascent until the rate stops improving, using a reasonable criteria.

---
## Step 3 — Response Surface Optimisation (1:30–3:00)

- [ ] Design a Latin-hypercube set of points around the best region found in Step 2 (`scipy.stats.qmc.LatinHypercube`), rpm still fixed
- [ ] Run it, fit a quadratic model (`rate ~ c + T + I(c**2) + I(T**2) + c:T`)
- [ ] Contour plot of predicted rate, with your best point and the model's predicted optimum marked (→ **Checkpoint 3b**)
- [ ] Confirm the predicted optimum with 1-2 extra experiments (an informal first check -- Step 4 makes this rigorous)

In [ ]:
# TODO: use qmc.LatinHypercube(d=2, seed=RANDOM_STATE) to generate sample
# points in [0,1]^2 for (c, T), then scale them to your chosen region.
# qmc.scale(sample, l_bounds, u_bounds) does the scaling for you. If you
# fixed your rpm value, rpm stays fixed at your chosen value for every point.

In [ ]:
# TODO:
# 1. run the LHS design and log() the results.
# 2. Fit the quadratic model on the FULL history (df_full = history) or
# just the RSM-region points: your call, but say which on the answer sheet.

In [ ]:
# TODO:
# 1. Build a grid over your explored (c, T) region. You will need
# to decide on the factor ranges, and the number of grid points.
# 2. Predict the rate with your fitted model at every grid point.
# 3. Make a contour plot using plt.contourf / ax.contour.
# Mark your best experimental point and the model's predicted optimum
# (e.g. from scipy.optimize.minimize on the negative of
# your fitted model, or just the grid's argmax).

In [ ]:
# TODO:
# 1. Store the predicted optimum (c_star, T_star, rate_star)
# -- you'll reuse it in Step 4.
# 2. Run 1-2 confirmation experiments at (or near) your predicted optimum
# and compare the observed rate to the model's prediction.

---
## Step 4 — How Sure Are You? A Confidence Interval (3:00–3:45)

- [ ] Run 8-10 replicate experiments at your predicted optimum (same T*, c*, and rpm every time).
- [ ] Compute the sample mean and standard deviation of the resulting rates
- [ ] Build a 95% confidence interval on the true mean rate there (→ **Checkpoint 4a**)
- [ ] Check whether the OVAT baseline (5.77 mol/s) and your model's predicted rate (Checkpoint 3b) fall inside or outside your CI
- [ ] Reflect on the sample-size / CI-width trade-off (→ **Checkpoint 4b**)

In [ ]:
# TODO: run 8-10 replicate experiments at (c_star, T_star), same settings
# every call. log() the results.

In [ ]:
# TODO: compute mean, standard deviation (ddof=1), and a 95% confidence
# interval on the mean rate. Use scipy.stats.t.ppf(0.975, df=n-1) for the
# critical value -- same one-sample CI construction as Part III.
# Then check: does 5.77 (OVAT) fall inside your CI? Does your Step 3
# predicted rate (rate_star) fall inside your CI?

In [ ]:
# TODO (Checkpoint 4b): using your own n and CI half-width, work out
# roughly how many replicates you'd need to halve the half-width
# (half-width shrinks with 1/sqrt(n)). Is that trade worth it given the
# contest rewards fewer experiments? Write your reasoning on the answer
# sheet, not just the number here.

---
## Wrap-up (3:45–4:00)

- [ ] Check your final experiment count and best rate with `leaderboard()`
- [ ] Compute your % improvement over the OVAT baseline of 5.77 mol/s
- [ ] Finalise your answer sheet (Checkpoints 1a–4c)

In [ ]:
# TODO: leaderboard() and a final summary print of best rate found,
# total experiments used, and % improvement over 5.77 mol/s.